# 🧬 Práctica B: Anotación de Genomas con conda en Google Colab

> **Antes de continuar**, lea la [guía de prácticas compartida](00_genome_annotation_common.md) — contiene el contexto biológico de cada caso, los conceptos clave, las plataformas y la descripción de los archivos de salida.

Esta práctica sigue el mismo flujo de trabajo que la **Práctica A** (Galaxy), pero ejecutando las herramientas desde la línea de comandos en un entorno Google Colab + conda. Esto le permite:

- Instalar y usar herramientas bioinformáticas de CLI sin necesitar un servidor institucional.
- Integrar los resultados directamente en Python con `pandas` y `matplotlib`.
- Comprender cómo se encadenan estas herramientas en un pipeline automatizado.

---

## ⚙️ Paso 0 — Configurar el entorno

Ejecute esta celda **primero**. Instala Miniconda y todas las herramientas necesarias.
⏱️ Puede tardar 8–12 minutos.

In [ ]:
import sys
!pip install condacolab
import condacolab
condacolab.install()

# Verify condacolab installation
import os
os.environ['PATH'] = '/usr/local/bin:' + os.environ['PATH']
!conda --version

Una vez que Conda esté instalado, es posible que necesites reiniciar el entorno de ejecución de Colab (Runtime > Restart runtime) para que los cambios surtan efecto. Después de reiniciar, puedes ejecutar comandos `conda` en nuevas celdas de código.

### Instalar herramientas

Ya que hemos instalado **Conda** es necesario instalar las herramientas que vamos a usar para el ensamblaje.

In [ ]:
# Instalar herramientas bioinformáticas
!conda install -y -c bioconda -c conda-forge \
    bakta \
    ncbi-amrfinderplus \
    plasmidfinder \
    integron_finder \
    isescan \
    antismash

!echo "✅ Instalación completa"

ahora vamos a instalar otras herramientas propias de **Python** que nos ayudaran a procesar y ver los resutlados mas facil para los humanos

In [ ]:
!pip install matplotlib pandas
!echo "✅ Instalación completa"

## 📥 Paso 1 — Descargar los datos del caso asignado

Solo trabajaremos con un solo caso, por lo cual debe definir primero el caso que le indicó el profesor (A, B o C). Asigne a la variable el caso que va a trabajar.

Consulte la [guía compartida](00_genome_annotation_common.md#-casos-de-estudio) para el contexto biológico de cada caso.

In [ ]:
# ⚠️ **IMPORTANTE**: Define el caso a trabajar aquí y solo aquí.
# Descomenta la línea que corresponda a tu caso (A, B o C).
CASO = "caso_C" # Por ejemplo, para caso A

In [ ]:
import os # Added import for os module

!export PATH="/opt/conda/bin:$PATH"

# Crear el directorio de datos para el caso seleccionado
!mkdir -p {CASO}/data

DOWNLOAD_SUCCESS = False

if CASO == "caso_A":
  print("── CASO A: Staphylococcus aureus MRSA (~2.8 Mb, GC ~33%) ──────────────────")
  !wget -q "https://zenodo.org/records/17252812/files/DRR187559_contigs.fasta" -O {CASO}/data/contigs.fasta
  DOWNLOAD_SUCCESS = True
elif CASO == "caso_B":
  print("── CASO B: Klebsiella pneumoniae (~5.5 Mb, GC ~57%) ───")
  !wget -q "https://zenodo.org/records/17252812/files/ERR14828471_contigs.fasta" -O {CASO}/data/contigs.fasta
  DOWNLOAD_SUCCESS = True
elif CASO == "caso_C":
  print("── CASO C: Streptomyces venezuelae (~8.2 Mb, GC ~72%) ───")
  !wget -q "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/253/235/GCF_000253235.1_ASM25323v1/GCF_000253235.1_ASM25323v1_genomic.fna.gz" -O {CASO}/data/contigs.fna.gz
  DOWNLOAD_SUCCESS = True
else:
  print("⚠️ Error: CASO no reconocido. Por favor, define 'CASO' como 'caso_A', 'caso_B' o 'caso_C'.")

# Descomprimir el genoma de referencia y mostrar mensaje de éxito si la descarga fue exitosa
if DOWNLOAD_SUCCESS:
  gz_file_path = f"{CASO}/data/contigs.fna.gz"
  fasta_file_path = f"{CASO}/data/contigs.fasta"

  if os.path.exists(gz_file_path):
    print(f"Descomprimiendo {gz_file_path}...")
    !gunzip {gz_file_path}
    # After unzipping, the file might be named contigs.fna if it was contigs.fna.gz
    # Ensure we rename it to contigs.fasta as per previous logic
    fna_path_after_unzip = f"{CASO}/data/contigs.fna"
    if os.path.exists(fna_path_after_unzip):
      !mv {fna_path_after_unzip} {fasta_file_path} # para mantener uniformidad
      print(f"Renombrado {fna_path_after_unzip} a {fasta_file_path}")

  if os.path.exists(fasta_file_path):
    # Use !grep to count contigs in the actual fasta file
    num_contigs_output = !grep -c '>' {fasta_file_path}
    num_contigs = num_contigs_output[0].strip()
    print(f"✅ Caso {CASO} listo — {num_contigs} contigs")
  else:
    print(f"⚠️ Error: No se pudo encontrar el archivo {fasta_file_path} después de la descarga/descompresión.")


## 👀 Paso 1.1 — Inspeccionar los archivos descargados

Vamos a verificar que los archivos FASTQ y el genoma de referencia FASTA se hayan descargado correctamente y tengan el formato esperado. Revisaremos el encabezado de cada uno.

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

print(f"\n=== Encabezado del archivo FASTA de contigs para {CASO} ===")
!head -n 10 {CASO}/data/contigs.fasta

## 🏷️ Paso 2 — Anotación principal con Bakta

Bakta realiza la anotación estructural (CDS, ARNt, ARNr, ncRNA, CRISPR) y funcional en un solo paso.

> **Nota:** Bakta requiere su base de datos. La descargamos en formato light (~1 GB) — suficiente para esta práctica.

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

# Descargar base de datos de Bakta (versión light)
!bakta_db download --output opt/bakta_db --type light
!echo "✅ Base de datos Bakta lista"

In [ ]:
import os

!export PATH="/opt/conda/bin:$PATH"

# Clear MPLBACKEND environment variable to avoid matplotlib backend conflicts
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']

!mkdir -p {CASO}/results/bakta_results

# ejecutar bakta
!bakta \
  --db opt/bakta_db/db-light \
  --output {CASO}/results/bakta_results \
  --force \
  --prefix genome \
  --keep-contig-headers \
  {CASO}/data/contigs.fasta

!echo "✅ Bakta completo. Resultados en ${CASO}/bakta_results/"

## 📊 Paso 3 — Explorar el resumen de Bakta con Python

In [ ]:
import pathlib, re

txt_path = pathlib.Path(f"{CASO}/results/bakta_results/genome.txt")

print(txt_path.read_text())

In [ ]:
# Extraer métricas clave del resumen
import pandas as pd

txt = txt_path.read_text()
metricas = {}
for linea in txt.splitlines():
    if ":" in linea:
        clave, _, valor = linea.partition(":")
        metricas[clave.strip()] = valor.strip()

df_resumen = pd.DataFrame.from_dict(metricas, orient="index", columns=["Valor"])
print(df_resumen.to_string())

### Visualización del `circular plot` generado por Bakta

In [ ]:
import os
from IPython.display import Image, display, SVG

# Construye la ruta al directorio de resultados de Bakta
bakta_results_dir = f"{CASO}/results/bakta_results"

# Lista los archivos en el directorio para encontrar el circular plot
print(f"Archivos en {bakta_results_dir}:")
!ls {bakta_results_dir}

# Intenta encontrar el archivo del circular plot (ahora buscamos .svg)
circular_plot_path = os.path.join(bakta_results_dir, "genome.svg")

if os.path.exists(circular_plot_path):
    print(f"\nMostrando el circular plot: {circular_plot_path}")
    display(SVG(filename=circular_plot_path))
else:
    print("\nNo se encontró el archivo 'genome.svg'. Verifique el directorio o el nombre del archivo.")

**Preguntas:**

1. ¿Cuántos contigs había en el input?
2. ¿Cuál es la longitud total del borrador del genoma (*draft genome length*)?
3. ¿Cuántos CDS fueron predichos?
4. ¿Cuántas *small proteins* se encontraron?
5. ¿Cuántos ARNt y ARNr se anotaron?
6. ¿Hay secuencias CRISPR?
7. Compare sus resultados con la **Tabla 1 de Hikichi et al. 2019** (Caso A) o el artículo de referencia (Caso B). ¿Coinciden?

## 🦠 Paso 4 — Genes de resistencia con AMRFinderPlus

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

# ORGANISMO="Staphylococcus_aureus" # Caso A
# ORGANISMO="Klebsiella_pneumoniae" # Caso B
# Caso C → comente la línea --organism (no existe grupo taxonómico para Streptomyces)

# Actualizar base de datos AMRFinder
!amrfinder -u
!echo "✅ Base de datos AMRFinder actualizada"

!mkdir -p {CASO}/results/amrfinder_results

# Redirigir la salida estándar de amrfinder al archivo para asegurar que se guarde.
!amrfinder \
  --nucleotide {CASO}/data/contigs.fasta \
  -o {CASO}/results/amrfinder_results/amrfinder.tsv  \
  # --organism {ORGANISMO} \
  --threads 2

!echo "✅ AMRFinderPlus completo"

print(f"\n--- Contenido de amrfinder.tsv (primeras 5 líneas) ---")
!head -n 5 {CASO}/results/amrfinder_results/amrfinder.tsv

In [ ]:
import pandas as pd

amr_df = pd.read_csv(f"{CASO}/results/amrfinder_results/amrfinder.tsv", sep="\t")
print(f"Total de genes encontrados: {len(amr_df)}\n")
print(amr_df[["Element symbol", "Contig id", "Class", "Subclass", "% Identity to reference"]].to_string())

**Preguntas:**

8. ¿Qué genes de resistencia a antibióticos se encontraron?
9. ¿A qué familias de antibióticos pertenecen (columna `Class`)?
10. ¿En qué contigs se encuentran?
11. ¿Se encontraron factores de virulencia?

## 🔵 Paso 5 — Identificación de plásmidos con PlasmidFinder

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

!mkdir -p {CASO}/results/plasmidfinder_results

!plasmidfinder.py \
  -i {CASO}/data/contigs.fasta \
  -o {CASO}/results/plasmidfinder_results \
  -p /usr/local/share/plasmidfinder-2.1.6/database \
  -x

!echo "✅ PlasmidFinder completo"

In [ ]:
import pandas as pd

pf_path = f"{CASO}/results/plasmidfinder_results/results_tab.tsv"
pf_df = pd.read_csv(pf_path, sep="\t")
print(f"Plásmidos encontrados: {len(pf_df)}\n")
print(pf_df[["Plasmid", "Identity", "Query / Template length", "Contig", "Position in contig"]].to_string())

**Preguntas:**

12. ¿Cuántos plásmidos se identificaron?
13. ¿A qué replicones / familias pertenecen?
14. ¿En qué contigs se encuentran?
15. Cruzando con AMRFinderPlus: ¿algún gen de resistencia está en el mismo contig que un plásmido?

## 🔗 Paso 6 — Detección de integrones con IntegronFinder

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

!mkdir -p {CASO}/results/integron_results

!integron_finder \
  --local-max \
  --func-annot {CASO}/data/contigs.fasta \
  --outdir {CASO}/results/integron_results

!echo "✅ IntegronFinder completo"

In [ ]:
import pathlib, pandas as pd

summary_path = list(pathlib.Path(f"{CASO}/results/integron_results").glob("*.summary"))[0]
print(summary_path.read_text())

**Preguntas:**

16. ¿Cuántos integrones completos se encontraron?
17. ¿Cuántos elementos In0 y CALIN?
18. ¿En qué contigs se localizan?

## 🔀 Paso 7 — Detección de elementos IS con ISEScan

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

!isescan.py \
  --seqfile {CASO}/data/contigs.fasta \
  --output {CASO}/results/isescan_results \
  --nthread 4

!echo "✅ ISEScan completo"

In [ ]:
import pandas as pd, pathlib, glob

is_files = glob.glob(f"{CASO}/results/isescan_results/*.tsv")
if is_files:
    is_df = pd.read_csv(is_files[0], sep="\t")
    print(f"Elementos IS detectados: {len(is_df)}\n")
    print(is_df[["family", "cluster", "seqID", "isBegin", "isEnd"]].to_string())
else:
    print("No se encontraron elementos IS o el archivo aún no está listo.")

**Preguntas:**

19. ¿Cuántos elementos IS se detectaron?
20. ¿En qué contigs se encuentran?
21. ¿Cuáles son las distintas familias de IS identificadas?

## 📊 Paso 8 — Resumen integrado con Python

Construya una tabla resumen que cruce los resultados de Bakta, AMRFinderPlus y PlasmidFinder por contig.

In [ ]:
import pandas as pd

# Cargar resultados
amr_df = pd.read_csv(f"{CASO}/results/amrfinder_results/amrfinder.tsv", sep="\t")
pf_df  = pd.read_csv(f"{CASO}/results/plasmidfinder_results/results_tab.tsv", sep="\t")

# Contigs con genes de resistencia
amr_contigs = set(amr_df["Contig id"].dropna().unique())
# Contigs con plásmidos
plasmid_contigs = set(pf_df["Contig"].dropna().unique())
# Intersección
both = amr_contigs & plasmid_contigs

print("=== Contigs con genes de resistencia ===")
print(sorted(amr_contigs))
print("\n=== Contigs con plásmidos ===")
print(sorted(plasmid_contigs))
print("\n=== Contigs con AMBOS (resistencia + plásmido) ===")
print(sorted(both) if both else "Ninguno")

## 🌿 Paso 9 — (Solo Caso C) Predicción de BGC con antiSMASH

> **Este paso es exclusivo para el Caso C (*Streptomyces venezuelae*)**. Los Casos A y B pueden saltar directamente a las preguntas integradoras.

Los clústeres de genes biosintéticos (BGC) son regiones del genoma que codifican la maquinaria completa para sintetizar un metabolito secundario — antibióticos, antifúngicos, sideróforos, pigmentos, etc. **antiSMASH** es la herramienta estándar para su predicción.

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

# Preparar base de datos de antiSMASH
!download-antismash-databases --database-dir opt/antismash_db

!echo "✅ Base de datos antiSMASH lista"

In [ ]:
!export PATH="/opt/conda/bin:$PATH"

!mkdir -p {CASO}/results/antismash_results

!antismash \
  {CASO}/data/contigs.fasta \
  --output-dir {CASO}/results/antismash_results \
  --taxon bacteria \
  --tfbs \
  --genefinding-tool prodigal \
  --pfam2go \
  --smcog-trees \
  --databases /content/opt/antismash_db \
  --cpus 4

!echo "✅ antiSMASH completo. Reporte en {CASO}/results/antismash_results/index.html"

### Explorar los resultados de antiSMASH con Python

In [ ]:
import json, pathlib, pandas as pd

json_path = list(pathlib.Path(f"{CASO}/results/antismash_results").glob("*.json"))[0]
data = json.loads(json_path.read_text())

# Extraer información de todos los BGC encontrados
registros = []
for record in data.get("records", []):
    for feature in record.get("features", []):
        if feature.get("type") == "region":
            quals = feature.get("qualifiers", {})
            registros.append({
                "Contig": record["id"],
                "BGC_type": ", ".join(quals.get("product", ["desconocido"])),
                "Start": feature["location"]["start"],
                "End": feature["location"]["end"],
                "Longitud_pb": feature["location"]["end"] - feature["location"]["start"],
                "KnownCluster_hit": quals.get("knownclusterblast", ["—"])[0][:60] if quals.get("knownclusterblast") else "—"
            })

bgc_df = pd.DataFrame(registros)
print(f"Total de BGC predichos: {len(bgc_df)}\n")
print(bgc_df.to_string())

In [ ]:
# Distribución de tipos de BGC
import matplotlib.pyplot as plt

tipo_counts = bgc_df["BGC_type"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
tipo_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Número de BGC")
ax.set_title(f"Tipos de BGC predichos en S. venezuelae ATCC 10712\n(antiSMASH, n={len(bgc_df)})")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{CASO}/results/antismash_results/bgc_tipos.png", dpi=150)
plt.show()
print("✅ Gráfico guardado")

**Preguntas — Caso C (antiSMASH):**

26. ¿Cuántos BGC predijo antiSMASH en el genoma de *S. venezuelae*?
27. ¿Qué tipos de BGC están presentes? ¿Cuál es el más frecuente?
28. ¿Cuál BGC corresponde al clúster del **cloranfenicol** según KnownClusterBlast?
29. ¿Hay BGC sin homología conocida en MIBiG? ¿Cuántos? ¿Qué implicaría caracterizar uno de ellos?
30. Elija un BGC que le llame la atención y describa: tipo, tamaño en pb y metabolito predicho.

## ❓ Preguntas integradoras

**Casos A y B:**

31. Considerando los resultados de Bakta, AMRFinderPlus y PlasmidFinder: ¿qué puede concluir sobre la movilidad de los genes de resistencia?
32. Si un gen de resistencia está en un plásmido que también contiene un integrón, ¿qué implica para la diseminación de resistencias?
33. ¿Todos los contigs con plásmidos pertenecen claramente al organismo de estudio? ¿Cómo lo verificaría?
34. Para el **Caso B** (*K. pneumoniae*): ¿los resultados de AMRFinderPlus son consistentes con el perfil de resistencia a carbapenemes descrito en el artículo?

**Caso C:**

35. ¿Cuántos BGC identificó Bakta en su anotación estándar? ¿Coincide con los predichos por antiSMASH? ¿Por qué podrían diferir?
36. Compare el número de BGC predichos con lo reportado en la literatura para *S. venezuelae* ATCC 10712. ¿Cuántos tienen producto conocido y cuántos son "silenciosos" o sin homología en MIBiG?
37. ¿Qué ventaja tiene usar un genoma *finished* (como GCF_000253235.1) en lugar de un borrador (*draft*) con muchos contigs para este tipo de análisis?

---
*Consulte la [guía compartida](00_genome_annotation_common.md) y el [README del Módulo 6](../README.md) para repasar los conceptos.*
